---
> 「言語とは、思考の衣装である。」 \
> (サミュエル・ジョンソン)
---

# GuppyLM ― たった5分でゼロから言語モデルを自作する

本ノートブックでは、Google Colab の無料枠でも約5分で訓練が完了する、GPT 風の小型言語モデル **GuppyLM** をゼロから実装する。

- nanoGPT (Karpathy, 2023) を参考に、Transformer デコーダのみの自己回帰モデルを PyTorch で構築する。
- データセットには tiny-shakespeare (約1MB) を用い、文字単位 (character-level) のトークナイザで学習する。
- 最終的に、シェイクスピア風のテキストを自動生成する。

GPT (Generative Pre-trained Transformer) は、OpenAI が 2018 年に発表して以来、ChatGPT (2022)、GPT-4 (2023)、GPT-5 系列 (2024-2025) と巨大化を続けてきた。しかし、その**核となるアーキテクチャは驚くほどシンプル**である。本ノートでは、その本質を 200 行程度のコードで体感する。

## 全体の流れ

1. **環境準備** ― PyTorch のインポートと GPU 設定
1. **データの取得** ― tiny-shakespeare コーパスのダウンロード
1. **トークナイザ** ― 文字単位での符号化・復号
1. **データセットとバッチ生成** ― ブロック単位のサンプリング
1. **モデル定義** ― Causal Self-Attention、Block、GuppyLM
1. **学習ループ** ― AdamW による最適化
1. **テキスト生成** ― 温度 (temperature) と top-k サンプリング
1. **評価とまとめ**

## 背景: なぜ GPT なのか

- 2017 年の Transformer 論文 (Vaswani et al.) 以降、自然言語処理の主流は注意機構ベースのモデルに置き換わった。
- BERT (2018) は Encoder のみを、GPT は **Decoder のみ**を用いる。
- GPT 系列は次のトークンを予測する**自己回帰 (autoregressive)** な目的関数だけで、驚くほど多様なタスクをこなせることが示された。
- 近年 (2024-2025) では、Llama 3 / 4、Mistral、Qwen 3、DeepSeek-V3、Gemma 3 など、オープンな GPT 系モデルが続々と公開されている。これらすべての**中核**は、本ノートで作る GuppyLM と本質的に同じである。

重要なのは、**規模を変えれば最先端モデルと地続きである**という事実を、自分の手で確認することである。

## 環境準備

必要なライブラリをインポートし、乱数シードと GPU を設定する。Colab の無料枠 (T4 GPU) でも問題なく動作する。

In [ ]:
import os
import math
import time
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## データの取得

tiny-shakespeare は Andrej Karpathy が公開した、シェイクスピア戯曲を結合した約 1MB のテキストファイルである。学習用コーパスとして広く用いられている。

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url, timeout=30).text
print(f"コーパスの長さ (文字数): {len(text):,}")
print("---- 先頭 250 文字 ----")
print(text[:250])

## トークナイザ (文字単位)

GPT-2 以降の実装では BPE (Byte Pair Encoding) が標準だが、ここでは**学習を 5 分で終えるため**最も単純な**文字単位**のトークナイザを採用する。

- コーパスに登場する一意な文字を語彙 (vocabulary) とする。
- 文字 → 整数 (`stoi`) と整数 → 文字 (`itos`) の対応表を作る。
- `encode` / `decode` で双方向に変換する。

文字単位は語彙サイズが小さく実装が簡単な反面、系列長が長くなる欠点がある。実用的な LLM では SentencePiece や tiktoken などのサブワード方式が使われる。

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"語彙サイズ: {vocab_size}")
print("文字一覧:", "".join(chars))

stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

def encode(s: str):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

# 動作確認
sample = "Hello, GuppyLM!"
ids = encode(sample)
print("encode:", ids)
print("decode:", decode(ids))

## データセットの分割とバッチ生成

テキスト全体を整数列に変換した後、先頭 90% を訓練、残り 10% を検証用に分割する。

GPT の学習では、長さ `block_size` の連続部分列をランダムに切り出してミニバッチを作る。入力 `x` に対する正解 `y` は、各位置の**次のトークン**である (1 文字ずらし)。

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"train: {len(train_data):,}  val: {len(val_data):,}")

block_size = 64   # 1 サンプルの文脈長
batch_size = 64   # ミニバッチサイズ

def get_batch(split: str):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - block_size - 1, (batch_size,))
    x = torch.stack([src[i : i + block_size] for i in ix])
    y = torch.stack([src[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("x shape:", xb.shape, " y shape:", yb.shape)

## モデル定義: Causal Self-Attention

GPT の心臓部は **Causal (Masked) Self-Attention** である。

- 入力 `x` から線形変換で Query / Key / Value を作る。
- 注意スコア $QK^\top / \sqrt{d_k}$ を計算する。
- **未来の位置を見てはいけない**ので、上三角行列を $-\infty$ でマスクする (causal mask)。
- softmax 後に Value と掛け、Multi-Head に分割して並列に処理する。

これが「次のトークンを予測する」自己回帰モデルの肝である。

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float = 0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        # 因果マスク (1 が許可、0 が未来)
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size),
        )

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)  # (B, T, 3C)
        q, k, v = qkv.chunk(3, dim=-1)
        # (B, n_head, T, head_dim) に変形
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        y = att @ v                                 # (B, n_head, T, head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))

## Transformer ブロック

1 つの **Block** は次の構成からなる。

1. LayerNorm → Causal Self-Attention → 残差接続
1. LayerNorm → Feed-Forward (2 層 MLP, 隠れ次元 4×) → 残差接続

GPT-2 以降では **Pre-LayerNorm** (注意機構の**前**に正規化) が採用されており、深いネットワークでも学習が安定する。GuppyLM もこの方式に従う。

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

## GuppyLM 本体

以下を順に積み上げる。

- **トークン埋め込み** (`tok_emb`): 整数 ID → ベクトル
- **位置埋め込み** (`pos_emb`): 系列内の位置 → ベクトル (学習可能)
- **N 段の Block** (Pre-LN Transformer)
- **最終 LayerNorm** と語彙への線形射影 (`lm_head`)

出力は各位置における次トークンの**ロジット**である。学習時はクロスエントロピーで、生成時は softmax 後にサンプリングする。

In [ ]:
class GuppyLM(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=4, block_size=64, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying (GPT-2 と同様)
        self.lm_head.weight = self.tok_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size
        pos = torch.arange(T, device=idx.device).unsqueeze(0)  # (1, T)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

## モデルの初期化

5 分以内で訓練が終わるよう、極めて小さな設定にする。

| ハイパーパラメータ | 値 |
|---|---|
| `n_layer` | 4 |
| `n_head` | 4 |
| `n_embd` | 128 |
| `block_size` | 64 |
| 総パラメータ数 | 約 0.2M |

なお、GPT-3 が 175B、GPT-4 系が推定 1T 規模と言われているのに対し、GuppyLM はその**約 100 万分の 1 〜 500 万分の 1**の規模である。それでも、シェイクスピアらしい英文を生成できる点が驚きである。

In [ ]:
model = GuppyLM(
    vocab_size=vocab_size,
    n_embd=128,
    n_head=4,
    n_layer=4,
    block_size=block_size,
    dropout=0.0,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"パラメータ数: {n_params/1e6:.3f} M")

## 学習ループ

AdamW で 2000 イテレーション学習する。Colab T4 GPU で約 3 〜 5 分で完了する。

- 一定間隔ごとに訓練・検証損失を表示する。
- 損失の値は**クロスエントロピー**で、ランダム初期化時は $\log(\text{vocab\_size}) \approx 4.17$ から始まる。
- うまくいけば、最終的に 1.5 前後まで下がる。

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=20):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

max_iters = 2000
eval_interval = 200
learning_rate = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start = time.time()
for it in range(max_iters + 1):
    if it % eval_interval == 0:
        losses = estimate_loss(model)
        elapsed = time.time() - start
        print(f"iter {it:5d} | train {losses['train']:.4f} | val {losses['val']:.4f} | {elapsed:6.1f}s")

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"総学習時間: {time.time() - start:.1f}s")

## テキスト生成

学習済みの GuppyLM に**プロンプト**を与え、自己回帰的に文字を生成する。

- **温度 (temperature)**: 値が小さいほど決定論的、大きいほど多様性が増す。
- **top-k**: 各ステップで上位 k 個の候補のみからサンプリングする。低品質な候補を抑制できる。

これらは ChatGPT などの API で使われている**サンプリング戦略**そのものである。

In [ ]:
prompt = "ROMEO:"
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
out = model.generate(context, max_new_tokens=400, temperature=0.9, top_k=40)
print(decode(out[0].tolist()))

## 温度を変えた比較

温度を変えると生成結果がどう変わるかを観察する。

In [ ]:
for temp in [0.4, 0.8, 1.2]:
    print(f"\n===== temperature = {temp} =====")
    context = torch.tensor([encode("JULIET:")], dtype=torch.long, device=device)
    out = model.generate(context, max_new_tokens=200, temperature=temp, top_k=40)
    print(decode(out[0].tolist()))

## 評価: パープレキシティ

言語モデルの評価指標として広く使われる **パープレキシティ (perplexity)** は、クロスエントロピー損失の指数で定義される。

$$ \text{PPL} = \exp(\mathcal{L}_{\text{CE}}) $$

値が小さいほど良いモデルである。

In [ ]:
losses = estimate_loss(model, eval_iters=50)
ppl_train = math.exp(losses["train"])
ppl_val = math.exp(losses["val"])
print(f"train loss: {losses['train']:.4f}  perplexity: {ppl_train:.2f}")
print(f"val   loss: {losses['val']:.4f}  perplexity: {ppl_val:.2f}")

## まとめ

- **約 200 行**で、GPT 風の自己回帰言語モデル GuppyLM をゼロから実装した。
- tiny-shakespeare で 5 分程度学習させるだけで、シェイクスピア風の英文を生成できることを確認した。
- 構成要素 (Causal Self-Attention、Pre-LayerNorm、残差接続、位置埋め込み、Weight Tying、AdamW、温度・top-k サンプリング) は、GPT-4 / Llama 4 / Gemma 3 / Qwen 3 など最新の LLM (2024-2025) と本質的に同一である。

### 次に試すこと

1. `n_layer`、`n_embd`、`block_size` を増やしてスケーリング則を体感する
1. 文字単位を BPE (例: `tiktoken`、`sentencepiece`) に置き換える
1. 日本語コーパス (青空文庫など) で学習する
1. RoPE や Flash-Attention、SwiGLU など最新の改良を組み込む
1. RLHF / DPO で指示追従性を高める

# 参考文献

- Vaswani, A. et al. (2017). "Attention Is All You Need." NeurIPS 2017.
- Radford, A. et al. (2018). "Improving Language Understanding by Generative Pre-Training." OpenAI Tech Report.
- Radford, A. et al. (2019). "Language Models are Unsupervised Multitask Learners." (GPT-2)
- Brown, T. et al. (2020). "Language Models are Few-Shot Learners." NeurIPS 2020. (GPT-3)
- Karpathy, A. (2023). "nanoGPT." https://github.com/karpathy/nanoGPT
- Touvron, H. et al. (2023-2024). "Llama 2 / Llama 3 / Llama 4." Meta AI.
- Jiang, A. Q. et al. (2023). "Mistral 7B." arXiv:2310.06825.
- DeepSeek-AI (2024). "DeepSeek-V3 Technical Report." arXiv:2412.19437.
- Google (2025). "Gemma 3 Technical Report."
- Qwen Team (2025). "Qwen 3 Technical Report."